In [1]:
import pandas as pd
import numpy as np
from langchain.document_loaders import PyPDFLoader
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import uuid
import json
import ollama.client as client



splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap  = 100,
    length_function = len,
    is_separator_regex = False,
)

In [2]:
from transformers import pipeline

## Roberta based NER
# ner = pipeline("token-classification", model="2rtl3/mn-xlm-roberta-base-named-entity", aggregation_strategy="simple")
ner = pipeline("token-classification", model="dslim/bert-large-NER", aggregation_strategy="simple")


print("Number of parameters ->", ner.model.num_parameters()/1000000, "Mn")


config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Number of parameters -> 332.538889 Mn


In [3]:
def row2NamedEntities(row):
    # print(row)
    ner_results = ner(row['text'])
    metadata = {'chunk_id': row['chunk_id']}
    entities = []
    for result in ner_results:
        entities = entities + [{'name': result['word'], 'entity': result['entity_group'], **metadata}]
        
    return entities

def dfText2DfNE(dataframe):
    ## Takes a dataframe from the parsed data and returns dataframe with named entities. 
    ## The input dataframe must have a text and a chunk_id column. 

    ## Using swifter for parallelism
    ## 1. Calculate named entities for each row of the dataframe. 
    results = dataframe.apply(row2NamedEntities, axis=1)

    ## Flatten the list of lists to one single list of entities. 
    entities_list = np.concatenate(results).ravel().tolist()

    ## Remove all NaN entities
    entities_dataframe = pd.DataFrame(entities_list).replace(' ', np.nan)
    entities_dataframe = entities_dataframe.dropna(subset=['entity'])

    ## Count the number of occurances per chunk id
    entities_dataframe = entities_dataframe.groupby(['name', 'entity', 'chunk_id']).size().reset_index(name='count')

    return entities_dataframe

In [5]:
loader = PyPDFLoader("/home/kartikey/project/knowledge_graph/data_input/cureus-0015-00000040274.pdf")
# loader = PyPDFDirectoryLoader("./data/kesy1dd")

pages = loader.load_and_split(text_splitter=splitter)
len(pages)


49

In [6]:

rows = []
for page in pages:
    row = {'text': page.page_content, **page.metadata, 'chunk_id': uuid.uuid4().hex}
    rows += [row]

df = pd.DataFrame(rows)


In [7]:
dfne = dfText2DfNE(df)


In [8]:
df_ne = dfne.groupby(['name', 'entity']).agg({'count': 'sum', 'chunk_id': ','.join}).reset_index()
df_ne.sort_values(by='count', ascending=False).head(100).reset_index()

,index,name,entity,count,chunk_id
0,37,India,LOC,28,"022cba816bb545b28516bcb1a5c3481d,05857bc14a5b4..."
1,56,Saxena,PER,6,"78e22dde8be44491a0115959d2fd2a64,970e7c22139f4..."
2,59,US,LOC,5,"5cda6bc75cf64be6b7e3f8f309d72b60,64651dddd5f44..."
3,14,AS,ORG,4,3331f39e7bf04a50bf29e17303d9325d
4,12,AN,ORG,4,3331f39e7bf04a50bf29e17303d9325d
...,...,...,...,...,...
63,30,Godfrey,PER,1,acc249d27ede4e69ba323f7318645607
64,31,Government of India,ORG,1,9270cea2c2d843eda272d56d8db7540c
65,32,Health,ORG,1,3239753e3276449fb9125764cea606cd
66,33,Health Centers,LOC,1,f951d7cdd8194c689e46ccd656a185f0


In [9]:
pages[12].page_content

'A key cadre has been the female community health workers, Accredited Social Health Activists (ASHA), onefor every 1,000 population. They number a million [11] to date and serve to increase the reach of theAuxiliary Nurse Midwives (ANM), who were meant to serve a population of 5,000 but in reality, serve up to20,000 people. This exercise in task shifting from ANMs to ASHAs cadre raises concerns. ANMs are highschool graduates who receive 18 months of training while ASHAs attend school up to the eighth grade,sometimes even less, and receive 23 days’ initial training with additional on-the-job need-based shorttraining [12]. ANMs are government employees and receive a salary while the ASHAs are treated as privatecontractors who receive a payment proportionate to the amount of work performed. A'

In [10]:


def extractConcepts(prompt: str, model='mistral-openorca:latest'):
    SYS_PROMPT = (
        "Your task is to extract the key entities mentioned in the users input.\n"
        "Entities may include - event, concept, person, place, object, document, organisation, artifact, misc, etc.\n"
        "Format your output as a list of json with the following structure.\n"
        "[{\n"
        "   \"entity\": The Entity string\n"
        "   \"importance\": How important is the entity given the context on a scale of 1 to 5, 5 being the highest.\n"
        "   \"type\": Type of entity\n"
        "}, { }]"
    )
    response, context = client.generate(model_name=model, system=SYS_PROMPT, prompt=prompt)
    return json.loads(response)


In [11]:
res = extractConcepts(prompt = pages[22].page_content)

Starting the generation process...
 [{"entity": "private sector", "importance": 3, "type": "organization"},
 {"entity": "medical colleges", "importance": 4, "type": "place"},
 {"entity": "India", "importance": 5, "type": "place"},
 {"entity": "May 2023", "importance": 3, "type": "time"},
 {"entity": "Saxena et al.", "importance": 4, "type": "person"},
 {"entity": "Cureus", "importance": 2, "type": "organization"},
 {"entity": "15(6): e40274", "importance": 1, "type": "misc"},
 {"entity": "DOI 10.7759/cureus.40274", "importance": 1, "type": "document"}]

In [12]:
res

[{'entity': 'private sector', 'importance': 3, 'type': 'organization'},
 {'entity': 'medical colleges', 'importance': 4, 'type': 'place'},
 {'entity': 'India', 'importance': 5, 'type': 'place'},
 {'entity': 'May 2023', 'importance': 3, 'type': 'time'},
 {'entity': 'Saxena et al.', 'importance': 4, 'type': 'person'},
 {'entity': 'Cureus', 'importance': 2, 'type': 'organization'},
 {'entity': '15(6): e40274', 'importance': 1, 'type': 'misc'},
 {'entity': 'DOI 10.7759/cureus.40274', 'importance': 1, 'type': 'document'}]